In [23]:
!pip install pymupdf
!pip install python-docx
#!pip install ollama
import os
import pandas as pd
import numpy as np
import fitz
from docx import Document
import json
import re
import requests
import joblib
from sklearn.metrics.pairwise import cosine_similarity

In [24]:
#Content Extration form the document
def read_txt(file_path):
    with open(file_path, "r", encoding="utf-8") as file:
        return file.read()

def read_pdf(file_path):
    document = fitz.open(file_path)
    text = ""
    for page in document:
       blocks = page.get_text("blocks")
       blocks = sorted(blocks,key=lambda block:(block[1],block[0]))
       for block in blocks:
          block_text = block[4].strip()
          if block_text:
              text += block_text + "\n"
       text += "\n"
    document.close()
    with open("Extracted_content_from_doc.txt",'w',encoding='utf-8') as file:
        file.write(text)
    #return text

def read_docx(file_path):
    document = Document(file_path)
    text = ""
    for paragraph in document.paragraphs:
        text += paragraph.text + "\n"
    return text

def read_csv(file_path):
    dataframe = pd.read_csv(file_path)
    return dataframe.to_string(index=False)

def read_excel(file_path):
    dataframe = pd.read_excel(file_path)
    return dataframe.to_string(index=False)

def read_document(file_path):
    extension = os.path.splitext(file_path)[1].lower()
    print(f"\nFile type detected: {extension}")
    if extension == ".txt":
        return read_txt(file_path)
    elif extension == ".pdf":
        return read_pdf(file_path)
    elif extension == ".docx":
        return read_docx(file_path)
    elif extension == ".csv":
        return read_csv(file_path)
    elif extension == ".xlsx":
        return read_excel(file_path)
    else:
        raise ValueError(
            f"Unsupported file type: {extension}"
        )

#read_document(r'/content/MY RESUME.pdf')
read_document(r'MY RESUME.pdf')


File type detected: .pdf


In [25]:
#Now Extracting the different sections from the extraxted text
SECTION_ALIASES = {
    "professional_summary": [
        "professional summary",
        "summary",
        "profile",
        "about me",
        "introduction",
        "career objective",
        "objective"
    ],

    "technical_skills": [
        "technical skills",
        "skills",
        "technical expertise",
        "core skills"
    ],

    "projects": [
        "projects",
        "academic projects",
        "personal projects"
    ],

    "education": [
        "education",
        "educational background",
        "academic background"
    ],

    "languages": [
        "languages",
        "language"
    ],

    "experience": [
        "experience",
        "work experience",
        "professional experience",
        "employment history"
    ],

    "internships": [
        "internships",
        "internship"
    ],

    "certifications": [
        "certifications",
        "certificates"
    ],

    "achievements": [
        "achievements",
        "awards",
        "honors"
    ]
}
'''SECTION_ALIASES = {
    "professional_summary": ["professional summary","summary","profile","about me","introduction","career objective","objective"],
    "technical_skills": ["technical skills","skills","technical expertise","core skills"],
    "projects": ["projects","academic projects","personal projects"],
    "education": ["education","educational background","academic background"],
    "languages": ["languages","language"],
    "experience": ["experience","work experience","professional experience","employment history"],
    "internships": ["internships","internship"],
    "certifications": ["certifications","certificates"],
    "achievements": ["achievements","awards","honors"]
}'''

def extract_sections(text):
    lines = text.splitlines()
    sections = {}
    current_section = None
    current_content = []
    for line in lines:
        line = line.strip()
        if not line:
            continue
        detected_section = detect_section(line)
        if detected_section:
            if current_section is not None:
                sections[current_section] = "\n".join(current_content).strip()
            current_section = detected_section
            current_content = []
        else:
            if current_section is not None:
                current_content.append(line)
    if current_section is not None:
        sections[current_section] = "\n".join(current_content).strip()
    return sections

def detect_section(line):
    normalized_line = normalize_heading(line)
    for section_name, aliases in SECTION_ALIASES.items():
        for alias in aliases:
            if normalized_line == normalize_heading(alias):
                return section_name
    return None

def normalize_heading(text):
    text = text.strip().lower()
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"[:\-]+$", "", text)
    return text.strip()

def save_json(data, output_file="resume_sections.json"):
    with open(output_file,"w",encoding="utf-8") as file:
        json.dump(data,file,indent=4,ensure_ascii=False)

input_file = "Extracted_content_from_doc.txt"
with open(input_file,"r",encoding="utf-8") as file:
    extracted_text = file.read()

sections = extract_sections(extracted_text)
save_json(sections)
print("\nSections extracted successfully!\n")

print(json.dumps(sections,indent=4,ensure_ascii=False))


Sections extracted successfully!

{
    "professional_summary": "Computer Science graduate with a blended skill set spanning data science and data analytics, including Python, SQL,\nmachine learning, and Power BI. Experienced in data cleaning, exploratory data analysis (EDA), predictive modeling, and\ndashboard development. Seeking an opportunity to apply both analytical and machine learning expertise to solve real-\nworld business problems.",
    "technical_skills": "Programming & Querying: Python Automation, SQL\nData Science & ML: NumPy, Pandas, Matplotlib, Seaborn, Scikit-learn, Predictive Modeling, EDA\nData Analytics & BI: Power BI Dashboard Development, Excel, KPI Reporting, Data Visualization\nTools & Platforms: Git, VS Code, Power BI Desktop, Postman, Jupyter Notebook, Google Colab",
    "projects": "House Price Predictor\n(Feb 2026)\nTech Stack: Python, Scikit-learn, Pandas, NumPy\n●\n●\nBuilt a machine learning regression model to predict house prices based on property feat

In [26]:
#now cleaning the resume sections to remove ("\n","●",spaces,extra lines)
def clean_text(text):
    text = text.replace("●", "")
    text = text.replace("\n", " ")
    text = re.sub(r"\s+", " ", text)
    text = text.strip()
    return text

def clean_sections(sections):
    with open(sections, "r", encoding="utf-8") as file:
        sections = json.load(file)
    cleaned_sections = {}
    for section, content in sections.items():
        cleaned_sections[section] = clean_text(content)
    return cleaned_sections

file_name = "resume_sections.json"
cleaned_data = clean_sections(file_name)

with open("cleaned_resume_sections.json","w",encoding='utf-8') as file:
    json.dump(cleaned_data,file,indent=4,ensure_ascii=False)

In [27]:
#creating chunks of the cleaned resume
def create_chunks(resume_data):
    chunks = []
    for section, content in resume_data.items():
        if not content:
            continue

        content = str(content).strip()
        if not content:
            continue
        chunk = {
            "text": content,
            "metadata": {
                "section": section,
                "chunk_type": section,
                "source": "resume"
            }
        }
        chunks.append(chunk)
    return chunks

with open("cleaned_resume_sections.json","r",encoding="utf-8") as file:
    resume_data = json.load(file)
chunks = create_chunks(resume_data)

with open("resume_chunks.json","w",encoding="utf-8") as file:
    json.dump(chunks,file,indent=4,ensure_ascii=False)

print(f"Total chunks created: {len(chunks)}")

for index, chunk in enumerate(chunks, start=1):
    print(f"\nChunk {index}")
    print(f"Section: {chunk['metadata']['section']}")
    print(f"Chunk Type: {chunk['metadata']['chunk_type']}")
    print(f"Text: {chunk['text'][:150]}...")

Total chunks created: 4

Chunk 1
Section: professional_summary
Chunk Type: professional_summary
Text: Computer Science graduate with a blended skill set spanning data science and data analytics, including Python, SQL, machine learning, and Power BI. Ex...

Chunk 2
Section: technical_skills
Chunk Type: technical_skills
Text: Programming & Querying: Python Automation, SQL Data Science & ML: NumPy, Pandas, Matplotlib, Seaborn, Scikit-learn, Predictive Modeling, EDA Data Anal...

Chunk 3
Section: projects
Chunk Type: projects
Text: House Price Predictor (Feb 2026) Tech Stack: Python, Scikit-learn, Pandas, NumPy Built a machine learning regression model to predict house prices bas...

Chunk 4
Section: education
Chunk Type: education
Text: B.Sc. in Computer Science | Mumbai University | CGPA: 8.67/10 | 2026 Bachelor's Degree HSC (Class XII), Science | Maharashtra State Board of Secondary...


In [28]:
#now going for embedding
def create_embedding(text_list):
    r = requests.post("http://localhost:11434/api/embed",json={
            "model": "bge-m3",
            "input": text_list
    })
    embedding = r.json()['embeddings']
    return embedding

json_file = "resume_chunks.json"
mydicts = []
chunk_id = 0

with open(json_file,"r",encoding="utf-8") as file:
    chunk = json.load(file)
    print("Embedding the document : ",json_file)
    embeddings = create_embedding([c["text"] for c in chunk])

    for i,chunk in enumerate(chunk):
        chunk['chunk_id'] = chunk_id
        chunk_id += 1
        chunk['embedding'] = embeddings[1]
        mydicts.append(chunk)

df = pd.DataFrame.from_records(mydicts)
print(df)
joblib.dump(df,'embedding.joblib')
df.to_csv("embedding.csv",index=False)

Embedding the document :  resume_chunks.json
                                                text  \
0  Computer Science graduate with a blended skill...   
1  Programming & Querying: Python Automation, SQL...   
2  House Price Predictor (Feb 2026) Tech Stack: P...   
3  B.Sc. in Computer Science | Mumbai University ...   

                                            metadata  chunk_id  \
0  {'section': 'professional_summary', 'chunk_typ...         0   
1  {'section': 'technical_skills', 'chunk_type': ...         1   
2  {'section': 'projects', 'chunk_type': 'project...         2   
3  {'section': 'education', 'chunk_type': 'educat...         3   

                                           embedding  
0  [-0.086154394, -0.008912311, -0.031846788, 0.0...  
1  [-0.086154394, -0.008912311, -0.031846788, 0.0...  
2  [-0.086154394, -0.008912311, -0.031846788, 0.0...  
3  [-0.086154394, -0.008912311, -0.031846788, 0.0...  


In [29]:
#now giving prompt and relevant context to the LLM
def inference(prompt):
    r = requests.post("http://localhost:11434/api/generate", json={
        # "model": "deepseek-r1",
        "model": "llama3.2",
        "prompt": prompt,
        "stream": False
    })
    response = r.json()
    print(response)
    return response

df = joblib.load('embedding.joblib')
incoming_query = input("Ask Me About Your Resume Or You Want Any Interview Preparation Assistant : ")
question_embedding = create_embedding([incoming_query])[0]

similarities = cosine_similarity(np.vstack(df['embedding']), [question_embedding]).flatten()
top_results = 5
max_indx = similarities.argsort()[::-1][0:top_results]
new_df = df.loc[max_indx]
context = "\n\n".join(new_df['text'].tolist())

#prompt 
prompt = f"""
You are an AI Interview Preparation Assistant.

The candidate has an upcoming interview for the following role:

Target Role:
{incoming_query}

Below is the candidate's resume information retrieved from the resume knowledge base.

Resume Context:
{context}

Your task is to generate interview preparation questions specifically
tailored to BOTH the target role and the candidate's resume.

Generate questions for these three rounds:

1. HR Round
2. Technical Round
3. Coding Round

IMPORTANT RULES:

- Use the candidate's resume as the primary source for understanding
  their skills, projects, education, and experience.
- Questions should be relevant to the target role.
- Ask questions about technologies, skills, and projects that actually
  appear in the resume.
- For projects, ask practical questions about implementation,
  architecture, challenges, decisions, and the candidate's contribution.
- Do not invent technologies, projects, experience, or achievements
  that are not present in the resume.
- If some information is not available in the resume, do not assume it.
- Coding questions should match the candidate's technical background
  and the requirements of the target role.
- HR questions should include resume-based questions such as:
  'Tell me about yourself', project discussions, career goals,
  strengths/weaknesses, and questions about the candidate's experience.
- Technical questions should cover concepts related to the skills
  and projects found in the resume.
- Make the questions progressively difficult.

Generate:

HR Round:
- 10 questions

Technical Round:
- 15 questions
  - 5 Easy
  - 5 Medium
  - 5 Hard

Coding Round:
- 10 questions
  - 3 Easy
  - 4 Medium
  - 3 Hard

For every question, provide:
- Question
- Difficulty
- Why the interviewer may ask this question

Do NOT provide answers unless explicitly requested by the user.

Target Role:
{incoming_query}
"""

response = inference(prompt)["response"]

{'model': 'llama3.2', 'created_at': '2026-09-19T10:27:11.6179466Z', 'response': "Here are the interview preparation questions tailored to the target role of Data Analysis, along with the required information:\n\n**HR Round:**\n\n1. Can you tell me about your educational background and how it has prepared you for a Data Analysis role?\n\t* Difficulty: Easy\n\t* Why: This question assesses the candidate's educational background and how it relates to the target role. It's a common icebreaker question that helps the interviewer understand the candidate's foundation.\n2. What motivates you to work in Data Analysis, and what do you hope to achieve in this role?\n\t* Difficulty: Easy\n\t* Why: This question helps the interviewer understand the candidate's motivations and goals. It's essential to understand what drives the candidate to apply for the role.\n3. Can you walk me through a time when you had to communicate complex data insights to a non-technical audience?\n\t* Difficulty: Medium\n\

In [31]:
#Recoding the prompt and the response by LLM
with open("prompt.txt",'w',encoding='utf-8') as file:
    file.write(prompt)

with open("response.txt",'w',encoding='utf-8') as file:
    file.write(response)